# Spark Streaming Energy Prediction

This notebook implements the real-time prediction layer of the building energy analytics project. The batch modelling notebook trained and saved a Spark ML model for 6-hour building-level energy consumption. This notebook loads that model, consumes weather events from Kafka, creates the same feature structure used during training, applies the model in streaming micro-batches, and publishes prediction outputs back to Kafka.

The streaming design follows a production-style pattern:

```text
weather_stream  →  Spark Structured Streaming  →  saved Spark ML model  →  prediction topics
```

The saved model predicts `log(1 + consumption_6h)`. This notebook converts predictions back to the original energy scale with `expm1(prediction_log)` before sending results to downstream topics.


## 1. Clear previous streaming state

A fresh streaming run should not share the session with old active streaming jobs. Stopping existing queries prevents duplicate consumption from Kafka and avoids writing repeated prediction records to the output topics.


In [3]:
# ==========================================
# 1. Stop Existing Streaming Queries
# ==========================================

try:
    active_queries = spark.streams.active

    if active_queries:
        for query in active_queries:
            query.stop()
        print("Existing streaming queries stopped:", len(active_queries))
    else:
        print("No active streaming queries found.")

except NameError:
    print("No Spark session exists yet.")

except Exception as exc:
    print("Unable to stop existing streams.")
    print(type(exc).__name__, str(exc))


No Spark session exists yet.


## 2. Start Spark with Kafka support

Spark is configured for local execution and includes the Kafka Structured Streaming connector. The connector is required because this notebook reads weather messages from Kafka and writes prediction outputs back to Kafka.


In [6]:
# ==========================================
# 2. Spark Session Setup
# ==========================================

import os
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark import SparkConf

try:
    spark.stop()
except NameError:
    pass
except Exception:
    pass

conf = (
    SparkConf()
    .setAppName("SparkStreamingEnergyPrediction")
    .setMaster("local[4]")
    .set("spark.driver.memory", "8g")
    .set("spark.executor.memory", "8g")
    .set("spark.executor.cores", "2")
    .set("spark.sql.shuffle.partitions", "8")
    .set("spark.default.parallelism", "8")
    .set("spark.sql.files.maxPartitionBytes", str(32 * 1024 * 1024))
    .set("spark.ui.showConsoleProgress", "false")
    .set(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
    )
)

spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark session created successfully.")
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Application name:", spark.sparkContext.appName)


:: loading settings :: url = jar:file:/opt/anaconda3/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/harshithreddy/.ivy2/cache
The jars for the packages stored in: /Users/harshithreddy/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9cb28e9a-37f6-446c-84be-c25e815c90e7;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 512ms :: artif

Spark session created successfully.
Spark version: 3.5.0
Spark master: local[4]
Application name: SparkStreamingEnergyPrediction


## 3. Load project paths

The streaming notebook uses the saved batch model and building metadata from the same project directory. Building metadata is required because incoming weather messages are site-level events, while the model produces building-level energy predictions.


In [9]:
# ==========================================
# 3. Imports and Project Paths
# ==========================================

import math
import json
import shutil
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType,
    StringType
)
from pyspark.ml import PipelineModel

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "spark_streaming_prediction_v2"

BUILDINGS_PATH = DATA_DIR / "building_information.csv"
MODEL_PATH = MODEL_DIR / "final_random_forest_model"
METADATA_PATH = MODEL_DIR / "model_metadata.json"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Building metadata path:", BUILDINGS_PATH)
print("Model path:", MODEL_PATH)
print("Model metadata path:", METADATA_PATH)
print("Checkpoint directory:", CHECKPOINT_DIR)

if not BUILDINGS_PATH.exists():
    raise FileNotFoundError(f"Missing building information file: {BUILDINGS_PATH}")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing saved model: {MODEL_PATH}")

print("Required project files found.")


Project root: /Users/harshithreddy/building-energy-prediction-streaming
Data directory: /Users/harshithreddy/building-energy-prediction-streaming/data
Building metadata path: /Users/harshithreddy/building-energy-prediction-streaming/data/building_information.csv
Model path: /Users/harshithreddy/building-energy-prediction-streaming/models/final_random_forest_model
Model metadata path: /Users/harshithreddy/building-energy-prediction-streaming/models/model_metadata.json
Checkpoint directory: /Users/harshithreddy/building-energy-prediction-streaming/checkpoints/spark_streaming_prediction_v2
Required project files found.


## 4. Configure Kafka topics

The producer notebook publishes weather observations to `weather_stream`. This notebook reads from that topic and writes three prediction outputs:

- `predictions_raw_v2`: detailed building-level prediction records
- `predictions_6h_v2`: one prediction per building and 6-hour window
- `predictions_daily_v2`: daily site-level totals rebuilt from 6-hour predictions

The `v2` suffix keeps the corrected log-target model outputs separate from earlier experimental output topics.


In [38]:
# ==========================================
# 4. Kafka and Runtime Configuration
# ==========================================

KAFKA_BROKER = "localhost:9092"

INPUT_WEATHER_TOPIC = "weather_stream"

PREDICTIONS_RAW_TOPIC = "predictions_raw_v2"
PREDICTIONS_6H_TOPIC = "predictions_6h_v2"
PREDICTIONS_DAILY_TOPIC = "predictions_daily_v2"

# Use "earliest" when the producer has already sent messages before this notebook starts.
# Use "latest" only when this notebook is already running before the producer starts.
STARTING_OFFSETS = "earliest"

# Keep False during setup. Change to True only when ready to start the streaming query.
START_STREAMING_JOB = True

# Reset checkpoints for a clean demo run.
RESET_CHECKPOINTS = True

print("Kafka broker:", KAFKA_BROKER)
print("Input weather topic:", INPUT_WEATHER_TOPIC)
print("Raw prediction topic:", PREDICTIONS_RAW_TOPIC)
print("6-hour prediction topic:", PREDICTIONS_6H_TOPIC)
print("Daily prediction topic:", PREDICTIONS_DAILY_TOPIC)
print("Starting offsets:", STARTING_OFFSETS)
print("Start streaming job:", START_STREAMING_JOB)
print("Reset checkpoints:", RESET_CHECKPOINTS)


Kafka broker: localhost:9092
Input weather topic: weather_stream
Raw prediction topic: predictions_raw_v2
6-hour prediction topic: predictions_6h_v2
Daily prediction topic: predictions_daily_v2
Starting offsets: earliest
Start streaming job: True
Reset checkpoints: True


## 5. Reset checkpoints for a clean run

Spark checkpoints store streaming progress. Resetting the checkpoint directory is useful after changing model logic because Spark can read the selected Kafka offsets again and write outputs using the corrected transformation.


In [15]:
# ==========================================
# 5. Reset Checkpoints
# ==========================================

if RESET_CHECKPOINTS and CHECKPOINT_DIR.exists():
    shutil.rmtree(CHECKPOINT_DIR)
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    print("Checkpoint directory reset.")
else:
    print("Checkpoint reset skipped.")

print("Checkpoint directory:", CHECKPOINT_DIR)


Checkpoint directory reset.
Checkpoint directory: /Users/harshithreddy/building-energy-prediction-streaming/checkpoints/spark_streaming_prediction_v2


## 6. Prepare Kafka output topics

The output topics are created before starting the stream. If they already exist, the notebook continues without changing them.


In [18]:
# ==========================================
# 6. Create Kafka Output Topics
# ==========================================

from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

def create_kafka_topics(topic_names, broker=KAFKA_BROKER):
    admin_client = None

    try:
        admin_client = KafkaAdminClient(
            bootstrap_servers=[broker],
            client_id="streaming_prediction_admin",
            request_timeout_ms=10000
        )

        existing_topics = admin_client.list_topics()

        topics_to_create = [
            NewTopic(
                name=topic_name,
                num_partitions=1,
                replication_factor=1
            )
            for topic_name in topic_names
            if topic_name not in existing_topics
        ]

        if topics_to_create:
            admin_client.create_topics(
                new_topics=topics_to_create,
                validate_only=False
            )
            print("Created topics:", [topic.name for topic in topics_to_create])
        else:
            print("All prediction output topics already exist.")

    except TopicAlreadyExistsError:
        print("One or more Kafka topics already exist.")

    except Exception as exc:
        print("Kafka topic setup failed.")
        print("Broker used:", broker)
        raise exc

    finally:
        if admin_client is not None:
            admin_client.close()


create_kafka_topics([
    PREDICTIONS_RAW_TOPIC,
    PREDICTIONS_6H_TOPIC,
    PREDICTIONS_DAILY_TOPIC
])


Created topics: ['predictions_raw_v2', 'predictions_6h_v2', 'predictions_daily_v2']


## 7. Load the saved model

The saved model includes the preprocessing pipeline and Random Forest regressor from the batch notebook. The streaming job recreates only the raw input feature columns expected by that pipeline.


In [21]:
# ==========================================
# 7. Load Saved Model and Metadata
# ==========================================

model = PipelineModel.load(str(MODEL_PATH))

print("Saved Spark ML model loaded successfully.")
print("Model path:", MODEL_PATH)

if METADATA_PATH.exists():
    with open(METADATA_PATH, "r") as f:
        model_metadata = json.load(f)

    print("\nModel metadata:")
    print(json.dumps(model_metadata, indent=4))
else:
    model_metadata = None
    print("No model metadata file found. Continuing with saved model only.")


Saved Spark ML model loaded successfully.
Model path: /Users/harshithreddy/building-energy-prediction-streaming/models/final_random_forest_model

Model metadata:
{
    "model_name": "Final Log-Target Random Forest",
    "target": "log_consumption_6h",
    "target_transformation": "log1p(consumption_6h)",
    "prediction_inverse_transformation": "expm1(prediction)",
    "model_grain": "site_id + building_id + 6-hour window",
    "rmsle": 1.2112076380234043,
    "sample_fraction": 0.1,
    "num_trees": 80,
    "max_depth": 12,
    "max_bins": 32,
    "categorical_features": [
        "primary_use",
        "season_flag"
    ],
    "numeric_features": [
        "log_square_feet",
        "floor_count",
        "building_age",
        "air_temperature_6h",
        "dew_temperature_6h",
        "sea_level_pressure_6h",
        "wind_speed_6h",
        "cloud_coverage_6h",
        "wind_dir_sin",
        "wind_dir_cos",
        "latent_y",
        "latent_s",
        "latent_r"
    ],
    "s

## 8. Prepare building metadata

Weather observations are site-level inputs. To produce building-level predictions, each 6-hour weather window is joined to all buildings at that site. Building attributes provide the static features required by the model, including building use, floor area, building age and floor count.


In [24]:
# ==========================================
# 8. Load and Clean Building Metadata
# ==========================================

buildings_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(BUILDINGS_PATH))
)

def standardise_column_names(df):
    for old_col in df.columns:
        new_col = (
            old_col
            .strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )
        df = df.withColumnRenamed(old_col, new_col)
    return df


buildings_df = standardise_column_names(buildings_df)

required_building_cols = [
    "building_id",
    "site_id",
    "primary_use",
    "square_feet"
]

missing_building_cols = [
    col_name for col_name in required_building_cols
    if col_name not in buildings_df.columns
]

if missing_building_cols:
    raise ValueError(f"Missing building columns: {missing_building_cols}")

if "year_built" not in buildings_df.columns:
    buildings_df = buildings_df.withColumn("year_built", F.lit(None).cast("double"))

if "floor_count" not in buildings_df.columns:
    buildings_df = buildings_df.withColumn("floor_count", F.lit(None).cast("double"))

buildings_clean = (
    buildings_df
    .withColumn("building_id", F.col("building_id").cast("int"))
    .withColumn("site_id", F.col("site_id").cast("int"))
    .withColumn("primary_use", F.coalesce(F.col("primary_use"), F.lit("Unknown")))
    .withColumn("square_feet", F.col("square_feet").cast("double"))
    .withColumn("year_built", F.col("year_built").cast("double"))
    .withColumn("floor_count", F.col("floor_count").cast("double"))
    .filter(F.col("building_id").isNotNull())
    .filter(F.col("site_id").isNotNull())
    .filter(F.col("square_feet").isNotNull())
    .filter(F.col("square_feet") > 0)
    .select(
        "building_id",
        "site_id",
        "primary_use",
        "square_feet",
        "year_built",
        "floor_count"
    )
    .cache()
)

print("Building metadata loaded and cleaned.")
print("Buildings available for prediction:", buildings_clean.count())

buildings_clean.orderBy("site_id", "building_id").show(10, truncate=False)


Building metadata loaded and cleaned.
Buildings available for prediction: 1000
+-----------+-------+-------------+-----------+----------+-----------+
|building_id|site_id|primary_use  |square_feet|year_built|floor_count|
+-----------+-------+-------------+-----------+----------+-----------+
|0          |0      |Education    |7432.0     |2015.0    |1.0        |
|1          |0      |Education    |2720.0     |2011.0    |1.0        |
|2          |0      |Education    |5376.0     |1998.0    |1.0        |
|3          |0      |Education    |23685.0    |2009.0    |1.0        |
|5          |0      |Education    |8000.0     |2007.0    |1.0        |
|6          |0      |Residential  |27926.0    |1988.0    |1.0        |
|8          |0      |Education    |60809.0    |2010.0    |1.0        |
|9          |0      |Office       |27000.0    |2017.0    |1.0        |
|10         |0      |Entertainment|370773.0   |1998.0    |1.0        |
|12         |0      |Residential  |37100.0    |2006.0    |1.0        

## 9. Define the incoming weather message structure

Each Kafka message from the producer is a JSON weather record. This schema converts the message into typed Spark columns that can be aggregated into the 6-hour windows used by the model.


In [27]:
# ==========================================
# 9. Define Weather Stream Schema
# ==========================================

weather_schema = StructType([
    StructField("site_id", IntegerType(), True),
    StructField("timestamp", StringType(), True),
    StructField("air_temperature", DoubleType(), True),
    StructField("cloud_coverage", DoubleType(), True),
    StructField("dew_temperature", DoubleType(), True),
    StructField("sea_level_pressure", DoubleType(), True),
    StructField("wind_direction", DoubleType(), True),
    StructField("wind_speed", DoubleType(), True)
])

print("Weather stream schema created.")


Weather stream schema created.


## 10. Define Kafka write utility

Each micro-batch creates static Spark DataFrames. This helper serialises each row to JSON and writes the records to the selected Kafka output topic.


In [30]:
# ==========================================
# 10. Helper Function to Write DataFrame to Kafka Topic
# ==========================================

def write_batch_to_kafka(df, topic_name):
    if df.rdd.isEmpty():
        print(f"No records to write to {topic_name}.")
        return

    if "building_id" in df.columns:
        key_col = F.col("building_id").cast("string")
    elif "site_id" in df.columns and "day" in df.columns:
        key_col = F.concat_ws("_", F.col("site_id").cast("string"), F.col("day").cast("string"))
    elif "site_id" in df.columns:
        key_col = F.col("site_id").cast("string")
    else:
        key_col = F.lit("prediction")

    kafka_df = (
        df
        .select(
            key_col.alias("key"),
            F.to_json(F.struct(*[F.col(c) for c in df.columns])).alias("value")
        )
    )

    (
        kafka_df
        .write
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BROKER)
        .option("topic", topic_name)
        .save()
    )

    print(f"Wrote {df.count()} records to topic: {topic_name}")


## 11. Build the micro-batch prediction function

The prediction logic is implemented with `foreachBatch` so each Kafka micro-batch can be processed like a normal Spark DataFrame. The function parses weather messages, aggregates weather observations to 6-hour site-level windows, joins building metadata, creates the same feature columns used in batch training, applies the saved model, converts the log prediction back to energy scale and writes prediction outputs to Kafka.


In [33]:
# ==========================================
# 11. Define Micro-Batch Prediction Function
# ==========================================

def process_weather_batch(batch_df, batch_id):
    print(f"\nProcessing batch_id: {batch_id}")

    if batch_df.rdd.isEmpty():
        print("Batch is empty.")
        return

    parsed_weather = (
        batch_df
        .select(
            F.from_json(
                F.col("value").cast("string"),
                weather_schema
            ).alias("data")
        )
        .select("data.*")
        .withColumn("site_id", F.col("site_id").cast("int"))
        .withColumn("weather_ts", F.to_timestamp(F.col("timestamp")))
        .filter(F.col("site_id").isNotNull())
        .filter(F.col("weather_ts").isNotNull())
        .cache()
    )

    weather_count = parsed_weather.count()
    print("Parsed weather records:", weather_count)

    if weather_count == 0:
        parsed_weather.unpersist()
        print("No valid weather records in batch.")
        return

    weather_6h = (
        parsed_weather
        .groupBy(
            "site_id",
            F.window("weather_ts", "6 hours").alias("time_window")
        )
        .agg(
            F.avg("air_temperature").alias("air_temperature_6h"),
            F.avg("dew_temperature").alias("dew_temperature_6h"),
            F.avg("sea_level_pressure").alias("sea_level_pressure_6h"),
            F.avg("wind_speed").alias("wind_speed_6h"),
            F.avg("cloud_coverage").alias("cloud_coverage_6h"),
            F.avg("wind_direction").alias("wind_direction_6h"),
            F.max("weather_ts").alias("event_time")
        )
        .withColumn("window_start", F.col("time_window.start"))
        .drop("time_window")
        .cache()
    )

    features_df = (
        weather_6h
        .join(buildings_clean, on="site_id", how="inner")
        .withColumn("log_square_feet", F.log1p(F.col("square_feet")))
        .withColumn(
            "building_age",
            F.when(
                F.col("year_built").isNotNull(),
                F.year(F.col("window_start")) - F.col("year_built")
            ).otherwise(None)
        )
        .withColumn("month", F.month(F.col("window_start")))
        .withColumn(
            "season_flag",
            F.when(F.col("month").isin([1, 2, 6, 7, 8, 12]), F.lit("peak"))
            .otherwise(F.lit("off_peak"))
        )
        .withColumn(
            "wind_direction_radians",
            F.col("wind_direction_6h") * F.lit(math.pi / 180.0)
        )
        .withColumn("wind_dir_sin", F.sin(F.col("wind_direction_radians")))
        .withColumn("wind_dir_cos", F.cos(F.col("wind_direction_radians")))
        .withColumn("day_of_year", F.dayofyear(F.col("window_start")))
        .withColumn("hour_of_day", F.hour(F.col("window_start")))
        .withColumn(
            "latent_y",
            F.sin(2 * F.lit(math.pi) * F.col("day_of_year") / F.lit(365.25))
        )
        .withColumn(
            "latent_s",
            F.cos(2 * F.lit(math.pi) * F.col("day_of_year") / F.lit(365.25))
        )
        .withColumn(
            "latent_r",
            F.sin(2 * F.lit(math.pi) * F.col("hour_of_day") / F.lit(24.0))
        )
        .select(
            "site_id",
            "building_id",
            "window_start",
            "event_time",
            "primary_use",
            "season_flag",
            "log_square_feet",
            "floor_count",
            "building_age",
            "air_temperature_6h",
            "dew_temperature_6h",
            "sea_level_pressure_6h",
            "wind_speed_6h",
            "cloud_coverage_6h",
            "wind_dir_sin",
            "wind_dir_cos",
            "latent_y",
            "latent_s",
            "latent_r"
        )
        .cache()
    )

    feature_count = features_df.count()
    print("Feature rows for prediction:", feature_count)

    if feature_count == 0:
        parsed_weather.unpersist()
        weather_6h.unpersist()
        features_df.unpersist()
        print("No feature rows created.")
        return

    scored_df = (
        model
        .transform(features_df)
        .withColumn(
            "prediction_log",
            F.greatest(F.col("prediction"), F.lit(0.0))
        )
        .withColumn(
            "predicted_energy_6h",
            F.expm1(F.col("prediction_log"))
        )
        .dropDuplicates(["site_id", "building_id", "window_start"])
        .cache()
    )

    raw_predictions_df = (
        scored_df
        .select(
            "site_id",
            "building_id",
            "window_start",
            "event_time",
            "primary_use",
            "season_flag",
            "prediction_log",
            "predicted_energy_6h"
        )
        .cache()
    )

    predictions_6h_df = (
        raw_predictions_df
        .select(
            "site_id",
            "building_id",
            "window_start",
            "event_time",
            "predicted_energy_6h"
        )
        .cache()
    )

    predictions_daily_df = (
        predictions_6h_df
        .withColumn("day", F.to_date("window_start"))
        .groupBy("site_id", "day")
        .agg(
            F.sum("predicted_energy_6h").alias("predicted_energy_daily"),
            F.countDistinct("building_id").alias("building_count")
        )
        .withColumn("event_time", F.current_timestamp())
        .select(
            "site_id",
            "day",
            "building_count",
            "predicted_energy_daily",
            "event_time"
        )
        .cache()
    )

    print("Raw prediction rows:", raw_predictions_df.count())
    print("6-hour prediction rows:", predictions_6h_df.count())
    print("Daily prediction rows:", predictions_daily_df.count())

    print("\nSample 6-hour predictions:")
    predictions_6h_df.orderBy("site_id", "building_id", "window_start").show(5, truncate=False)

    write_batch_to_kafka(raw_predictions_df, PREDICTIONS_RAW_TOPIC)
    write_batch_to_kafka(predictions_6h_df, PREDICTIONS_6H_TOPIC)
    write_batch_to_kafka(predictions_daily_df, PREDICTIONS_DAILY_TOPIC)

    parsed_weather.unpersist()
    weather_6h.unpersist()
    features_df.unpersist()
    scored_df.unpersist()
    raw_predictions_df.unpersist()
    predictions_6h_df.unpersist()
    predictions_daily_df.unpersist()

    print(f"Batch {batch_id} completed.")


## 12. Create the streaming source

The stream reads weather messages from Kafka. `STARTING_OFFSETS` controls whether Spark reads from the earliest available messages or only new messages arriving after the stream starts.


In [36]:
# ==========================================
# 12. Read Weather Stream from Kafka
# ==========================================

weather_stream = (
    spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("subscribe", INPUT_WEATHER_TOPIC)
    .option("startingOffsets", STARTING_OFFSETS)
    .load()
)

print("Weather stream DataFrame created.")
print("Input topic:", INPUT_WEATHER_TOPIC)
print("Starting offsets:", STARTING_OFFSETS)


Weather stream DataFrame created.
Input topic: weather_stream
Starting offsets: earliest


## 13. Start the streaming prediction job

Keep `START_STREAMING_JOB = False` while checking setup cells. When Kafka is running and the producer messages are available, change it to `True` in the configuration cell and run this cell to start the query.


In [40]:
# ==========================================
# 13. Start Streaming Prediction Job
# ==========================================

if START_STREAMING_JOB:
    prediction_query = (
        weather_stream
        .writeStream
        .foreachBatch(process_weather_batch)
        .option("checkpointLocation", str(CHECKPOINT_DIR / "weather_prediction_query"))
        .start()
    )

    print("Streaming prediction query started.")
    print("Query ID:", prediction_query.id)

else:
    print("Streaming prediction job is not started.")
    print("Set START_STREAMING_JOB = True in Cell 4 when ready.")


Streaming prediction query started.
Query ID: 85636b4d-da44-4728-bc71-ffcb003d07cc


## 14. Monitor streaming progress

This cell confirms whether the streaming query is active and whether Spark is processing Kafka messages. During a successful run, the status changes from waiting to processing when data is available.


In [55]:
# ==========================================
# 14. Check Active Streaming Queries
# ==========================================

active_queries = spark.streams.active

print("Active streaming queries:", len(active_queries))

for query in active_queries:
    print("Query ID:", query.id)
    print("Name:", query.name)
    print("Status:", query.status)
    print("Is active:", query.isActive)

    if query.lastProgress:
        print("\nLast progress:")
        print(query.lastProgress)
    else:
        print("\nNo progress yet.")

    print("-" * 60)


Active streaming queries: 1
Query ID: 85636b4d-da44-4728-bc71-ffcb003d07cc
Name: None
Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}
Is active: True

Last progress:
{'id': '85636b4d-da44-4728-bc71-ffcb003d07cc', 'runId': '89653192-d15d-4e8f-88dd-1758da90de98', 'name': None, 'timestamp': '2026-09-08T11:54:48.906Z', 'batchId': 6, 'numInputRows': 200, 'inputRowsPerSecond': 27.586206896551722, 'processedRowsPerSecond': 38.37298541826554, 'durationMs': {'addBatch': 5127, 'commitOffsets': 35, 'getBatch': 0, 'latestOffset': 9, 'queryPlanning': 3, 'triggerExecution': 5212, 'walCommit': 37}, 'stateOperators': [], 'sources': [{'description': 'KafkaV2[Subscribe[weather_stream]]', 'startOffset': {'weather_stream': {'0': 6900}}, 'endOffset': {'weather_stream': {'0': 7000}}, 'latestOffset': {'weather_stream': {'0': 7000}}, 'numInputRows': 200, 'inputRowsPerSecond': 27.586206896551722, 'processedRowsPerSecond': 38.37298541826554, 'metrics': {'avg

## 15. Stop the streaming job after the demo

Run this cell only after the prediction messages have been created and the consumer visualisation notebook has consumed the output topics.


In [57]:
# ==========================================
# 15. Stop Streaming Queries
# ==========================================

for query in spark.streams.active:
    query.stop()

print("All active streaming queries stopped.")


All active streaming queries stopped.


## Run checklist

1. Keep Docker, Zookeeper and Kafka running.
2. Run this notebook through Cell 12.
3. In Cell 4, set `START_STREAMING_JOB = True`.
4. Run Cell 13 to start the prediction query.
5. Run Cell 14 to confirm the stream is active.
6. If no data is processed, rerun the producer notebook or keep `STARTING_OFFSETS = "earliest"` with reset checkpoints enabled.
7. Use the consumer visualisation notebook to read `predictions_raw_v2`, `predictions_6h_v2` and `predictions_daily_v2`.
8. Run Cell 15 only when the full streaming demo is complete.
